# Esercitazione

Importa le diverse librerie

In [ ]:
import random
import re
from collections import Counter, defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## Preparazione dati e analisi del testo

Importa i dati da un archivio
dei contenuti di Wikipedia e messo a disposizione sulla piattaforma
[Hugging Face](https://huggingface.co/datasets/Salesforce/wikitext) (Merity et al., 2016).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Utilizza prima solo i dati del dataset *test*. In un secondo momento puoi ripetere l'esercitazione con il dataset *train*, cambiando il nome tra parentesi quadre.

In [ ]:
data = dataset["test"]
testo = " ".join(data["text"])
print("Lunghezza testo:", len(testo))

Lunghezza testo: 1289979


### Tokenizzazione

Crea una lista delle parole presenti.

In [ ]:
tokens = re.findall(r"\b\w+\b", testo.lower())
tokens[:10]

['robert',
 'boulter',
 'robert',
 'boulter',
 'is',
 'an',
 'english',
 'film',
 'television',
 'and']

## Modello di ordine 0



### Training

Cosa apprende il modello? Nella fase di training calcola le frequenza di ogni **parola tipo** presente nel testo. Il dataframe *df* creato contiene le informazioni apprese da nostro semplice modello di liguaggio naturale.

In [ ]:
tokens = re.findall(r"\b\w+\b", testo.lower())
l_corpus= len(tokens)
frequenze_type = Counter(tokens)

df = (
    pd.DataFrame(frequenze_type.items(), columns=["type", "frequenza"])
    .sort_values(by="frequenza", ascending=False)
    .reset_index(drop=True)
)
df["probabilità"] = df.frequenza/l_corpus
df.head()

,type,frequenza,probabilità
0,the,16083,0.077891
1,of,6789,0.032880
2,and,5885,0.028502
3,in,5079,0.024598
4,to,4787,0.023184


### Predict

Dopo la fase di apprendimento puoi passare a fare la previsione. In questo caso il modello ti restituisce semplicemente la parola più probabile.

In [ ]:
def predictM0(df):
  parola_successiva = df.loc[df["probabilità"].idxmax(), "type"]
  return ("La parola prevista è:", parola_successiva)

In [ ]:
predictM0(df)

('La parola prevista è:', 'the')

### Osservazioni

Cosa non considera un modello di ordine 0?

Inserisci qui le tue risposte

## Modello di ordine 1

Considera ora la probabilità di ogni bigramma e non più della singola parola type.

### Training

Costruisci i bigrammi cioè le coppie di parole

In [ ]:
bigrammi = list(zip(tokens[:-1], tokens[1:]))
print(bigrammi[1:10])

[('boulter', 'robert'), ('robert', 'boulter'), ('boulter', 'is'), ('is', 'an'), ('an', 'english'), ('english', 'film'), ('film', 'television'), ('television', 'and'), ('and', 'theatre')]


Calcola la frequenza di ogni bigramma tipo, quindi dell'evento intersezione.

In [ ]:
frequenze_bigrammi = Counter(bigrammi)
print(frequenze_bigrammi)

Counter({('of', 'the'): 2187, ('in', 'the'): 1390, ('to', 'the'): 811, ('and', 'the'): 558, ('on', 'the'): 522, ('for', 'the'): 429, ('at', 'the'): 388, ('by', 'the'): 351, ('from', 'the'): 345, ('with', 'the'): 345, ('as', 'a'): 305, ('the', 'first'): 305, ('as', 'the'): 302, ('to', 'be'): 264, ('of', 'a'): 249, ('in', 'a'): 241, ('it', 'was'): 236, ('that', 'the'): 232, ('he', 'was'): 231, ('during', 'the'): 198, ('with', 'a'): 184, ('was', 'the'): 173, ('was', 'a'): 149, ('and', 'a'): 148, ('to', 'a'): 147, ('the', 'city'): 145, ('one', 'of'): 140, ('of', 'his'): 136, ('the', 'film'): 136, ('had', 'been'): 118, ('have', 'been'): 117, ('is', 'a'): 115, ('it', 'is'): 112, ('km', 'h'): 111, ('the', 'north'): 110, ('is', 'the'): 109, ('due', 'to'): 106, ('and', 'was'): 106, ('after', 'the'): 106, ('the', 'second'): 103, ('the', 'british'): 101, ('number', 'of'): 100, ('for', 'a'): 99, ('that', 'he'): 99, ('the', 'united'): 99, ('such', 'as'): 98, ('the', 'philippines'): 98, ('he', 'had'

Costruisci una tabella con le seguenti colonne: parola successiva $v_2$, parola precedente $v_1$ e la probabilità condizionata di $v_2$ nota la parola precedente $v_1$.

In [ ]:
righe = []

for (w1, w2), count in frequenze_bigrammi.items():
    freq_w1 = frequenze_type[w1]
    prob_w1 = freq_w1/l_corpus
    prob_w1_et_w2 = count / (l_corpus-1)
    righe.append([w1, w2, prob_w1_et_w2, prob_w1, count/freq_w1])

df_bigrammi = pd.DataFrame(
    righe,
    columns=[
        "w1",
        "w2",
        "P(w1_et_w2)",
        "P(w1)",
        "P(w2|w1)"
    ]
)

df_bigrammi.head()

,w1,w2,P(w1_et_w2),P(w1),P(w2|w1)
0,robert,boulter,0.000029,0.000097,0.300000
1,boulter,robert,0.000005,0.000116,0.041667
2,boulter,is,0.000005,0.000116,0.041667
3,is,an,0.000155,0.005507,0.028144
4,an,english,0.000010,0.003511,0.002759


### Predict
Genera la parola successiva in base alla probabilità condizionata.

In [49]:
def predictM1(parola, df_bigrammi):
    if parola not in df_bigrammi.w1.values:
        return df_parola.loc[df_parola["P(w1)"].idxmax()].w1

    # Trova la parola con probabilità massima
    df_parola = df_bigrammi[df_bigrammi.w1==parola]
    succ = df_parola.loc[df_parola["P(w2|w1)"].idxmax()].w2
    return succ


In [63]:
def predictM1(parola, df_bigrammi):
    # Se la parola non è nel dizionario, scegliamo una parola a caso basandoci sulle frequenze generali
    if parola not in df_bigrammi.w1.values:
        return np.random.choice(df_bigrammi.w1.values)

    df_possibili = df_bigrammi[df_bigrammi.w1 == parola]
    opzioni = df_possibili.w2.values
    probabilita = df_possibili["P(w2|w1)"].values
    succ = np.random.choice(opzioni, p=probabilita)

    return succ

In [66]:
print(predictM1("the", df_bigrammi))

teams


### L'autoregressione

Proviamo a generare un testo di n parole.

In [67]:
def genera_testo_markov1(parola_iniziale, n_parole, corpus):
    parola = parola_iniziale
    risultato = [parola]

    for _ in range(n_parole - 1):
        next_word = predictM1(parola, corpus)
        if next_word is None:
            break
        risultato.append(next_word)
        parola = next_word

    return " ".join(risultato)


In [68]:
print(genera_testo_markov1("swim", 12, df_bigrammi))

swim through may 12 55 he is neither discrimination nor the lrt


### Osservazioni

Cosa osservi? Il modello ha dimenticato l'inizio della frase?

Inserisci la tua risposta.

# Fonti

Linguistica computazionale, A. Lenci, S. Auriemma, M. Miliani, Hoeppli